# CR Algorithm Test

## Useful Functions

In [486]:
import numpy as np

from pathlib import Path
import sys
parent_dir = Path.cwd().parent.resolve()
sys.path.append(str(parent_dir))

np.set_printoptions(precision=3)

In [487]:
from numpy.linalg import norm
from numpy import sin, cos, tan, sqrt, pi
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass
import pytz

In [488]:
mu_unicode = "\u03bc"
deg_unicode = "\u00b0"

In [489]:
def circle_points(radius, N = 100, position = None):
    p = np.array([0,0]) if position is None else position
    t = np.linspace(0, pi * 2, N, endpoint=False)
    # print(p)
    # print(radius)
    # print(cos(t).shape)
    # print(radius * cos(t) + p[0])
    return np.array([radius * cos(t) + p[0], radius * sin(t) + p[1]])
    # x = radius * np.cos(t) + p[0]
    # y = radius * np.sin(t) + p[1]
    # return np.vstack((x, y)).T   # shape (N,2)

def noise(cov):
    return np.random.multivariate_normal(
        mean = np.zeros(cov.shape[0]),
        cov=cov
    )

R_example = np.array([
    [0.5, 0, 0],
    [0, 0.5, 0],
    [0, 0, 0]
])
# print(f"Example noise: {noise(R_example)}\nwith R = \n{R_example}")

## Setup

In [490]:
import trajectory as T
from PlanetaryData import Earth, Luna, Sun
from Spice import SpiceImporter
from algorithms import ChristianRobinson
from Constants import RAD_TO_DEG, DEG_TO_RAD, ARCSEC_TO_RAD, RAD_TO_ARCSEC

## Camera

In [491]:
# # Camera specs 
# # e.g. https://spinworks.pt/products/star-tracker-st1/

# # Inputs
# fov_diag = 18.2 * DEG_TO_RAD
# f = 50e-3               # focal length
# n_pixels = (2048, 2048)
# mu = 5.5e-6             # pixel spacing [m]
# sigma = 2 * ARCSEC_TO_RAD / 3 # Gives in 3-sigma

# # Calculated
# mu_angle = mu / f       # pixel angle resolution [rad]
# print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcseconds")
# fov = fov_diag / np.sqrt(2)
# print(f"FOV: {fov*RAD_TO_DEG:.2f}º")

In [492]:
# Camera specs 
# e.g. https://satsearch.co/products/arcsec-sagitta-star-tracker?utm_source=chatgpt.com

camera_model = "sagitta"

if camera_model == "sagitta":
    # Inputs
    fov = 25.2 * DEG_TO_RAD
    f = 2500e-3               # focal length
    n_pixels = np.array([2048, 2048])   # Assumed from previous star tracker

    cross_boresight_sigma = 2 * ARCSEC_TO_RAD
    boresight_sigma = 10 * ARCSEC_TO_RAD 
    # Calculated
    mu_angle = fov/n_pixels[0]       # pixel angle resolution [rad/px]
    mu = f * mu_angle             # pixel spacing [m/px]

    
print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcs/px")
print(f"Pixel resolution: {mu*1000000:.2f} {mu_unicode}m/px")
print(f"FOV: {fov*RAD_TO_DEG:.2f}{deg_unicode}")

Pixel anglular resolution: 44.30 arcs/px
Pixel resolution: 536.89 μm/px
FOV: 25.20°


In [493]:
class Camera:
    def __init__(self,
        f: float,
        mu: float,
        resolution: float
    ):
        """Camera object

        Args:
            f (float): Focal length [m]
            mu (float): Pixel resolution [m/px]
            resolution (np.ndarray(2)): Total pixels (x, y) [px]
        """
        self.f = f
        self.mu = mu
        self.resolution = resolution

        self.mu_angle = mu/f
        self.fov = self.resolution * self.mu_angle

        self.are_specs_calc = False

    # mu_angle = fov/n_pixels[0]       # pixel angle resolution [rad/px]
    # mu = f * mu_angle             # pixel spacing [m/px]
    def calc_specs(self, alpha_skew: float = 0):

        dx = dy = self.f / self.mu
        u_p, v_p = self.resolution / 2

        self.K_inv = np.array([
            [1/dx, -alpha_skew/dx/dy, (alpha_skew*v_p - dy*u_p)/dx/dy],
            [0, 1/dy, -v_p/dy],
            [0, 0, 1]
        ])

        self.u_p = u_p
        self.v_p = v_p
        self.dx = dx
        self.dy = dy

        self.are_specs_calc = True
        

    def __str__(self):
        s = "Camera base specs:\n"
        s += f" - Pixel angular resolution: {self.mu_angle*RAD_TO_ARCSEC:.2f} arcs/px\n"
        s += f" - Pixel resolution ({mu_unicode}): {self.mu*1000000:.2f} {mu_unicode}m/px\n"
        s += f" - FOV: {self.fov*RAD_TO_DEG}{deg_unicode}\n"

        if self.are_specs_calc:
            s += " Calculated specs\n"
            s += f" - Center (u_p, v_p) = {np.array([self.u_p, self.v_p])} arcs/px\n"
            s += f" - dx, dy = {np.array([self.dx, self.dy])} (unitless?)\n"
            # np.set_printoptions(formatter={'all': lambda x: '\t' + str(x)})
            s += f" - K_inv = \n{self.K_inv}\n"
            # s += f" - Pixel resolution: {self.mu*1000000:.2f} {mu_unicode}m/px\n"
            # s += f" - FOV: {self.fov*RAD_TO_DEG}{deg_unicode}\n"

        return s

In [494]:
class Body:

    def __init__(self,
        a: float,
        b: float,
        c: float
    ):
        """
        Args:
            a (float): Ellipsoid parameter [m]
            b (float): Ellipsoid parameter [m]
            c (float): Ellipsoid parameter [m]
        """
        self.a = a
        self.b = b
        self.c = c

    def __str__(self):
        s = "Worldview:\n"
        s += f" - Ellipsoid semi-axes a,b,c = [{self.a}, {self.b}, {self.c}] m\n"
        return s
    
class Pose:
    def __init__(self,
        r_truth: np.ndarray,
#         a: float,
#         b: float,
#         c: float,
        T_p_c: np.ndarray
    ):
            
        """

        Args:
            r_truth (np.ndarray): Position relative to central body [m]
            T_p_c (np.ndarray(3,3)): Passive rotation from PLANET to CAMERA 
        """
        self.r_truth = r_truth
        self.T_p_c = T_p_c

    def __str__(self):
        s = "Pose:\n"
        s += f" - Spacecraft position (r_truth) = {self.r_truth} m\n"
        s += f" - Absolute position norm = {norm(self.r_truth):.2f} m\n"
        return s

In [495]:
class PlanetImage:
    def __init__(self, c: Camera, b: Body, sigma_cross_boresight_px: float = 0.5):
        """Init and generate planet points

        Args:
            c (Camera): Camera
            b (Body): Body specs with ellipsoid parameters
            sigma_cross_boresight_px (float, optional): Standard deviation [px]. Defaults to 0.5.
        """
        self.camera_center_px = np.array([c.u_p, c.v_p])
        self.camera_mu_angle = c.mu_angle
        self.res = c.resolution

        # Noise
        s = sigma_cross_boresight_px
        self.R = np.array([
            [s**2,0,0],
            [0,s**2,0],
            [0,0,0]
        ])

        # TODO: change if not circular, or if manually specifying center and radius
        self.dx = c.dx # no idea what this is really
        self.body_radius = b.a

        self.image_generated = False

    def gen_points(self, pose: Pose, N_RADIAL_POINTS: int = 100,
                 body_center_px: np.ndarray = None, body_radius_px: float = None):
        """Generate planet points based on position

        Args:
            pose (Pose): Pose with position and orientation
            N_PLANET_POINTS (int): Number of planet points to generate 
            planet_center_px (np.ndarray(2), optional): Planet center [px]. Defaults to image center.
            planet_radius_px (float, optional): Planet radius [px]. Only if you want to manually specify.
        """
        self.image_generated = True


        # Planet pixel locations in image 
        circular = True
        
        self.body_angle = np.arcsin(self.body_radius / norm(pose.r_truth)  )

        if circular: # TODO: change this for non circular or offset
            # self.body_angle = np.arcsin(self.body_radius / norm(pose.r_truth))
            
            self.body_center_px = (
                self.camera_center_px if body_center_px is None else body_center_px
            )

        else:
            # From 
            #   -   planet->camera in camera frame
            #   -   camera->planet in camera frame (for image generation)
            planet_vector_cam = pose.T_p_c @ -pose.r_truth 

            Xc, Yc, Zc = planet_vector_cam

            # projection: u = f * Xc / Zc / mu → in pixels
            u = (Xc / Zc) / self.camera_mu_angle + self.camera_center_px[0]
            v = (Yc / Zc) / self.camera_mu_angle + self.camera_center_px[1]

            self.body_center_px = np.array([u, v])

        # self.body_radius_px = (
        #         self.body_angle / self.camera_mu_angle if body_radius_px is None else body_radius_px
        #     )
        self.body_radius_px = self.dx * tan(self.body_angle)

        # Generate points
        self.N = N_RADIAL_POINTS
        self.body_points = circle_points(self.body_radius_px, self.N, # RETURNS (2, N) ARRAY
                                    position=self.body_center_px) 
        


        self.u_points = np.hstack((self.body_points.T, np.ones((N_RADIAL_POINTS, 1))))
        self.u_noise = np.array([v + noise(self.R) for v in self.u_points])
        self.u_in_frame = np.array([v for v in self.u_noise 
                       if v[0] > 0 and v[0] < self.res[0]
                       and v[1] > 0 and v[1] < self.res[1]])
    

    def __str__(self):
        s = "PlanetImage:\n"
        s += f" - Noise covariance R [px^2]= \n{self.R}\n"
        s += f" - Planet center = {self.camera_center_px} px\n"
        
        if self.image_generated:
            s += " Current image specs:\n"
            s += f" - Planet angular radius (rad) = {self.body_angle:.6f} rad\n"
            s += f" - Planet radius = {self.body_radius_px:.2f} px\n"
            s += f" - Number of points = {self.N}\n"
            s += f" - First 3 noisy points = \n{self.u_noise[:3]} px\n"

        return s

In [496]:
# One-time calculation
camera = Camera(f, mu, n_pixels)
camera.calc_specs()
body = Body(a=3000e3, b=3000e3, c=3000e3)
print(camera)
print(body)

cr_alg = ChristianRobinson(camera.K_inv, body.a, body.b, body.c)

Camera base specs:
 - Pixel angular resolution: 44.30 arcs/px
 - Pixel resolution (μ): 536.89 μm/px
 - FOV: [25.2 25.2]°
 Calculated specs
 - Center (u_p, v_p) = [1024. 1024.] arcs/px
 - dx, dy = [4656.419 4656.419] (unitless?)
 - K_inv = 
[[ 2.148e-04  0.000e+00 -2.199e-01]
 [ 0.000e+00  2.148e-04 -2.199e-01]
 [ 0.000e+00  0.000e+00  1.000e+00]]

Worldview:
 - Ellipsoid semi-axes a,b,c = [3000000.0, 3000000.0, 3000000.0] m



In [497]:
def run_step(position, orientation, offset = None , print_stats = False, plot_points = False):

    # Calculate each time there is a new image
    pose = Pose(position, orientation)
    img = PlanetImage(camera, body, 0.5)

    if offset:
        offset = np.asarray(offset)
    img.gen_points(pose, N_RADIAL_POINTS=100, body_center_px=offset)
    # print(pose)
    # print(img)

    # Run algorithm
    rc = cr_alg.run(img.u_noise, pose.T_p_c)

    # # DEBUG: Print some values
    # print(f"True position: {pose.r_truth}")
    # print(f"Body center in image: {img.body_center_px}")
    # print(f"Body radius in pixels: {img.body_radius_px}")
    # print(f"Body angular radius: {img.body_angle}")
    # print(f"First 3 u_noise points:\n{img.u_noise[:3]}")


    # Print statistics
    if print_stats:
        with np.printoptions(precision=2, suppress=True):
            print(f"CR Position: {rc} ({norm(rc)})m")
            print("CR stats:")
            print(f"- Absolute error: {(rc - pose.r_truth)} m ({norm(rc - pose.r_truth):.2f}m)")
        print(f"- Relative error: {abs(rc - pose.r_truth)/norm(pose.r_truth)} \n")

    if plot_points:
        print(img.res)
        # print(img.u_noise)
        plt.scatter(img.u_in_frame[:,0], img.u_in_frame[:,1])
        # plt.scatter(img.u_noise[:,0], img.u_noise[:,1])
        plt.xlim((0, camera.resolution[0]))
        plt.ylim((0, camera.resolution[1]))
        plt.show()

    # print(img)

In [498]:
run_step(np.array([0, 0, 17000e3]), np.eye(3), print_stats=True)

CR Position: [    -274.35       61.56 17000125.7 ] (17000125.70259172)m
CR stats:
- Absolute error: [-274.35   61.56  125.7 ] m (307.99m)
- Relative error: [1.614e-05 3.621e-06 7.394e-06] 



In [499]:
# 90 degree rotation because z axis of camera is along y axis
R = np.array([
    [0,1,0],
    [0,0,-1],
    [-1,0,0]
])

def make_T_p_c_lookat(r_truth, up_p=np.array([0.0, 0.0, 1.0])):
    # desired camera +Z (in planet frame): point from spacecraft to planet
    z_p = -r_truth / norm(r_truth)

    # camera +X (in planet frame): right = up × z
    x_p = np.cross(up_p, z_p)
    if norm(x_p) < 1e-9:  # if up is parallel to z, pick another up
        up_p = np.array([0.0, 1.0, 0.0])
        x_p = np.cross(up_p, z_p)
    x_p /= norm(x_p)

    # camera +Y (down) (in planet frame)
    y_p = np.cross(z_p, x_p)

    # columns are camera axes expressed in planet frame => maps planet vec -> camera vec via transpose
    C_p = np.column_stack((x_p, y_p, z_p))
    return C_p.T  # this is T_p_c
T_p_c = make_T_p_c_lookat(np.array([21000e3, 0, 0]))

r = np.array([21000e3, 100000, 0])
run_step(r, R, print_stats=True)

print(norm(r))

CR Position: [     128.89     -434.54 21000412.52] (21000412.52767144)m
CR stats:
- Absolute error: [-20999871.11   -100434.54  21000412.52] m (29698855.20m)
- Relative error: [1.    0.005 1.   ] 

21000238.093888365


In [500]:
print(R @ [1,0,0])
print(R @ [0,1,0])
print(R @ [0,0,1])
print(R@[0, 7000e3, 0])

[ 0  0 -1]
[1 0 0]
[ 0 -1  0]
[7000000.       0.       0.]
